In [1]:
import torch ## torch let's us create tensors and also provides helper functions
import torch.nn as nn ## torch.nn gives us nn.Module(), nn.Embedding() and nn.Linear()
import torch.nn.functional as F # This gives us the softmax() and argmax()
from torch.optim import Adam ## We will use the Adam optimizer, which is, essentially,
                             ## a slightly less stochastic version of stochastic gradient descent.
from torch.utils.data import TensorDataset, DataLoader ## We'll store our data in DataLoaders

import lightning as L

In [6]:
## first, we create a dictionary that maps vocabulary tokens to id numbers...
token_to_id = {'what' : 0,
               'is' : 1,
               'statquest' : 2,
               'awesome': 3,
               '<EOS>' : 4, ## <EOS> = end of sequence
              }
## ...then we create a dictionary that maps the ids to tokens. This will help us interpret the output.
## We use the "map()" function to apply the "reversed()" function to each tuple (i.e. ('what', 0)) stored
## in the token_to_id dictionary. We then use dict() to make a new dictionary from the
## reversed tuples.
id_to_token = dict(map(reversed, token_to_id.items()))

## NOTE: Because we are using a Decoder-Only Transformer, the inputs contain
##       the questions ("what is statquest?" and "statquest is what?") followed
##       by an <EOS> token followed by the response, "awesome".
##       This is because all of those tokens will be used as inputs to the Decoder-Only
##       Transformer during Training. (See the illustration above for more details)
## ALSO NOTE: When we train this way, it's called "teacher forcing".
##       Teacher forcing helps us train the neural network faster.
inputs = torch.tensor([[token_to_id["what"], ## input #1: what is statquest <EOS> awesome
                        token_to_id["is"],
                        token_to_id["statquest"],
                        token_to_id["<EOS>"],
                        token_to_id["awesome"]],

                       [token_to_id["statquest"], # input #2: statquest is what <EOS> awesome
                        token_to_id["is"],
                        token_to_id["what"],
                        token_to_id["<EOS>"],
                        token_to_id["awesome"]]])

## NOTE: Because we are using a Decoder-Only Transformer the outputs, or
##       the predictions, are the input questions (minus the first word) followed by
##       <EOS> awesome <EOS>.  The first <EOS> means we're done processing the input question
##       and the second <EOS> means we are done generating the output.
##       See the illustration above for more details.
labels = torch.tensor([[token_to_id["is"],
                        token_to_id["statquest"],
                        token_to_id["<EOS>"],
                        token_to_id["awesome"],
                        token_to_id["<EOS>"]],

                       [token_to_id["is"],
                        token_to_id["what"],
                        token_to_id["<EOS>"],
                        token_to_id["awesome"],
                        token_to_id["<EOS>"]]])

## Now let's package everything up into a DataLoader...
dataset = TensorDataset(inputs, labels)
dataloader = DataLoader(dataset)

In [7]:
class PositionEncoding(nn.Module):

    def __init__(self, d_model=2, max_len=6):
        """
        d_model: int
            Dimensionality of embeddings (features per token).

        max_len: int
            Maximum sequence length supported.
        """
        super().__init__()

        # Initialize positional encoding matrix
        # Shape: (max_len, d_model)
        positional_encoding = torch.zeros(max_len, d_model)

        # Create position indices [0, 1, ..., max_len-1]
        # Shape: (max_len, 1)
        position_indices = torch.arange(
            start=0,
            end=max_len,
            step=1
        ).float().unsqueeze(1)

        # Create embedding dimension indices for even positions (0, 2, 4, ...)
        # Shape: (d_model/2,)
        dimension_indices = torch.arange(
            start=0,
            end=d_model,
            step=2
        ).float()

        # Compute scaling factor for each dimension
        # Formula: 1 / (10000^(i / d_model))
        # Controls frequency of sine/cosine waves
        scaling_factor = 1 / torch.tensor(10000.0) ** (
            dimension_indices / d_model
        )

        # Apply sine to even dimensions (0, 2, 4, ...)
        positional_encoding[:, 0::2] = torch.sin(
            position_indices * scaling_factor
        )

        # Apply cosine to odd dimensions (1, 3, 5, ...)
        positional_encoding[:, 1::2] = torch.cos(
            position_indices * scaling_factor
        )

        # Register as buffer:
        # - Not trainable
        # - Moves with model (CPU/GPU)
        self.register_buffer('pe', positional_encoding)

    def forward(self, word_embeddings):
        """
        word_embeddings: Tensor
            Shape: (sequence_length, d_model)

        Returns:
            Position-aware embeddings of same shape.
        """

        # Add positional encoding corresponding to sequence length
        # Ensures shape compatibility
        return word_embeddings + self.pe[:word_embeddings.size(0), :]

In [8]:
class Attention(nn.Module):
    def __init__(self, d_model=2):
        """
        d_model: int
            Dimensionality of token embeddings (features per token).
        """
        super().__init__()

        # Linear projections to generate Query (Q), Key (K), and Value (V)
        # Each maps from embedding space → embedding space
        self.query_projection = nn.Linear(d_model, d_model, bias=False)
        self.key_projection   = nn.Linear(d_model, d_model, bias=False)
        self.value_projection = nn.Linear(d_model, d_model, bias=False)

        # Define which dimensions represent rows and columns
        # For input shape: (sequence_length, d_model)
        self.row_dim = 0   # tokens (positions)
        self.col_dim = 1   # embedding features

    def forward(self, query_input, key_input, value_input, mask=None):
        """
        query_input, key_input, value_input: Tensor
            Shape: (sequence_length, d_model)

        mask: Tensor or None
            Shape: (sequence_length, sequence_length)
            True values indicate positions to ignore (mask out).
        """

        # Step 1: Project inputs into Q, K, V spaces
        Q = self.query_projection(query_input)   # (seq_len, d_model)
        K = self.key_projection(key_input)       # (seq_len, d_model)
        V = self.value_projection(value_input)   # (seq_len, d_model)

        # Step 2: Compute raw attention scores
        # Formula: Q × K^T
        # Result shape: (seq_len, seq_len)
        attention_scores = torch.matmul(
            Q,
            K.transpose(self.row_dim, self.col_dim)
        )

        # Step 3: Scale attention scores
        # Formula: scores / sqrt(d_model)
        d_k = Q.size(self.col_dim)
        scaled_scores = attention_scores / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

        # Step 4: Apply mask (if provided)
        # Masked positions get large negative value → softmax ≈ 0
        if mask is not None:
            scaled_scores = scaled_scores.masked_fill(mask, -1e9)

        # Step 5: Convert scores to probabilities
        # Softmax over keys dimension (columns)
        attention_weights = F.softmax(scaled_scores, dim=self.col_dim)

        # Step 6: Weighted sum of values
        # Output shape: (seq_len, d_model)
        attention_output = torch.matmul(attention_weights, V)

        return attention_output

In [9]:
class DecoderOnlyTransformer(L.LightningModule):

    def __init__(self, num_tokens=4, d_model=2, max_len=6):
        """
        num_tokens: int
            Vocabulary size.

        d_model: int
            Embedding dimension (features per token).

        max_len: int
            Maximum sequence length supported.
        """
        super().__init__()

        # Set seed for reproducibility (consistent weight initialization)
        L.seed_everything(seed=42)

        # Step 1: Token embedding layer
        # Maps token IDs → dense vectors
        # Input:  (seq_len,)
        # Output: (seq_len, d_model)
        self.token_embedding = nn.Embedding(
            num_embeddings=num_tokens,
            embedding_dim=d_model
        )

        # Step 2: Positional encoding
        # Injects order information into embeddings
        self.position_encoding = PositionEncoding(
            d_model=d_model,
            max_len=max_len
        )

        # Step 3: Masked self-attention
        # Enables each token to attend to previous tokens only
        self.self_attention = Attention(d_model=d_model)

        # Step 4: Output projection layer
        # Maps from embedding space → vocabulary logits
        # Output: (seq_len, num_tokens)
        self.output_projection = nn.Linear(
            in_features=d_model,
            out_features=num_tokens
        )

        # Loss function (applies softmax internally)
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, token_ids):
        """
        token_ids: Tensor
            Shape: (sequence_length,)

        Returns:
            logits: Tensor
            Shape: (sequence_length, num_tokens)
        """

        # Step 1: Convert token IDs → embeddings
        embeddings = self.token_embedding(token_ids)

        # Step 2: Add positional encoding
        position_encoded_embeddings = self.position_encoding(embeddings)

        # Step 3: Create causal (look-ahead) mask
        seq_len = token_ids.size(0)

        # Lower triangular matrix (allowed positions)
        causal_mask = torch.tril(torch.ones((seq_len, seq_len)))

        # Convert to boolean mask:
        # True  → masked (future positions)
        # False → allowed
        causal_mask = causal_mask == 0

        # Step 4: Masked self-attention
        # Q = K = V = position_encoded_embeddings
        attention_output = self.self_attention(
            position_encoded_embeddings,
            position_encoded_embeddings,
            position_encoded_embeddings,
            mask=causal_mask
        )

        # Step 5: Residual connection
        decoder_state = position_encoded_embeddings + attention_output

        # Step 6: Project to vocabulary logits
        logits = self.output_projection(decoder_state)

        return logits

    def configure_optimizers(self):
        # Adam optimizer (learning rate should typically be smaller, e.g., 1e-3)
        return Adam(self.parameters(), lr=0.1)

    def training_step(self, batch, batch_idx):
        """
        batch: tuple(input_tokens, target_tokens)

        input_tokens: Tensor (batch_size, seq_len)
        target_tokens: Tensor (batch_size, seq_len)
        """

        input_tokens, target_tokens = batch

        # Remove batch dimension (assuming batch_size = 1)
        input_sequence = input_tokens[0]
        target_sequence = target_tokens[0]

        # Forward pass
        logits = self.forward(input_sequence)

        # Compute loss
        # logits: (seq_len, vocab_size)
        # target: (seq_len,)
        loss = self.criterion(logits, target_sequence)

        return loss

In [10]:
# Step 0: Initialize model
model = DecoderOnlyTransformer(
    num_tokens=len(token_to_id),
    d_model=2,
    max_len=6
)

# Step 1: Prepare initial input sequence
# Example: "what is statquest <EOS>"
input_token_ids = torch.tensor([
    token_to_id["what"],
    token_to_id["is"],
    token_to_id["statquest"],
    token_to_id["<EOS>"]
])

# Current sequence length
input_sequence_length = input_token_ids.size(0)

# Step 2: Forward pass through model
# Output: logits for each position → (seq_len, vocab_size)
logits = model(input_token_ids)

# Step 3: Select next token using greedy decoding
# - Take last position (latest token prediction)
# - Apply argmax over vocabulary dimension
next_token_id = torch.argmax(logits[-1, :]).unsqueeze(0)

# Initialize generated sequence with first predicted token
generated_token_ids = next_token_id

# Step 4: Autoregressive generation loop
max_length = 6

for step in range(input_sequence_length, max_length):

    # Stop if <EOS> token is generated
    if next_token_id.item() == token_to_id["<EOS>"]:
        break

    # Append predicted token to input sequence
    input_token_ids = torch.cat((input_token_ids, next_token_id))

    # Recompute logits for updated sequence
    logits = model(input_token_ids)

    # Predict next token from last position
    next_token_id = torch.argmax(logits[-1, :]).unsqueeze(0)

    # Append to generated output sequence
    generated_token_ids = torch.cat(
        (generated_token_ids, next_token_id)
    )

# Step 5: Convert predicted IDs → tokens
print("Predicted Tokens:\n")
for token_id in generated_token_ids:
    print("\t", id_to_token[token_id.item()])

Seed set to 42


Predicted Tokens:

	 <EOS>


In [15]:
trainer = L.Trainer(max_epochs=30, accelerator= 'cpu')
trainer.fit(model, train_dataloaders=dataloader)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name              | Type             | Params | Mode  | FLOPs
-----------------------------------------------------------------------
0 | token_embedding   | Embedding        | 10     | train | 0    
1 | position_encoding | PositionEncoding | 0      | train | 0    
2 | self_attention    | Attention        | 12     | train | 0    
3 | output_projection | Linear           | 15     | train | 0    
4 | criterion         | CrossEntropyLoss | 0      | train | 0    
----------------

Epoch 29: 100%|█| 2/2 [00:00<00:00, 628.74it/s

`Trainer.fit` stopped: `max_epochs=30` reached.


Epoch 29: 100%|█| 2/2 [00:00<00:00, 395.20it/s


In [16]:

# Step 1: Prepare initial input sequence
# Example: "what is statquest <EOS>"
input_token_ids = torch.tensor([
    token_to_id["what"],
    token_to_id["is"],
    token_to_id["statquest"],
    token_to_id["<EOS>"]
])

# Current sequence length
input_sequence_length = input_token_ids.size(0)

# Step 2: Forward pass through model
# Output: logits for each position → (seq_len, vocab_size)
logits = model(input_token_ids)

# Step 3: Select next token using greedy decoding
# - Take last position (latest token prediction)
# - Apply argmax over vocabulary dimension
next_token_id = torch.argmax(logits[-1, :]).unsqueeze(0)

# Initialize generated sequence with first predicted token
generated_token_ids = next_token_id

# Step 4: Autoregressive generation loop
max_length = 6

for step in range(input_sequence_length, max_length):

    # Stop if <EOS> token is generated
    if next_token_id.item() == token_to_id["<EOS>"]:
        break

    # Append predicted token to input sequence
    input_token_ids = torch.cat((input_token_ids, next_token_id))

    # Recompute logits for updated sequence
    logits = model(input_token_ids)

    # Predict next token from last position
    next_token_id = torch.argmax(logits[-1, :]).unsqueeze(0)

    # Append to generated output sequence
    generated_token_ids = torch.cat(
        (generated_token_ids, next_token_id)
    )

# Step 5: Convert predicted IDs → tokens
print("Predicted Tokens:\n")
for token_id in generated_token_ids:
    print("\t", id_to_token[token_id.item()])

Predicted Tokens:

	 awesome
	 <EOS>


📘 Technical Report: Decoder-Only Transformer Architecture

⸻

1. Overview

A decoder-only transformer is a neural architecture designed for autoregressive sequence modeling, where the objective is to predict the next token given all previous tokens:

P(x_t \mid x_{<t})

This architecture underlies modern language models (e.g., GPT) and operates using masked self-attention without a separate encoder.

⸻

2. Core Components

2.1 Token Embedding

Maps discrete token IDs to continuous vectors:

\text{Embedding}: \mathbb{Z} \rightarrow \mathbb{R}^{d_{model}}

Output shape:
(seq\_len, d_{model})

⸻

2.2 Positional Encoding

Since attention is permutation-invariant, positional encoding injects order:

x = \text{Embedding} + \text{Positional Encoding}

Using sinusoidal functions:

PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)
PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)

⸻

2.3 Masked Self-Attention

Core mechanism enabling contextualization while preserving causality.

Attention function:

\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V

In decoder-only:

Q = K = V = x

⸻

2.4 Causal Mask

Ensures that each token attends only to previous tokens:

\text{mask}_{i,j} =
\begin{cases}
0 & j \le i \\
-\infty & j > i
\end{cases}

Implemented via lower triangular matrix.

⸻

2.5 Residual Connection

y = x + \text{Attention}(x)

Purpose:

* preserves input signal
* stabilizes gradient flow

⸻

2.6 Output Projection

Maps hidden states to vocabulary logits:

\text{logits} = xW + b

Shape:
(seq\_len, vocab\_size)

⸻

2.7 Loss Function

Cross-entropy loss:

\mathcal{L} = -\log P(y_{true})

Softmax is applied internally.

⸻

3. Forward Pass Pipeline

Token IDs
   ↓
Embedding Layer
   ↓
+ Positional Encoding
   ↓
Masked Self-Attention
   ↓
+ Residual Connection
   ↓
Linear Projection
   ↓
Logits

⸻

4. Training Procedure

4.1 Objective

Train the model to predict next token:

x_1, x_2, ..., x_n \rightarrow x_2, x_3, ..., x_{n+1}

⸻

4.2 Input–Target Alignment

Input Sequence	Target Sequence
[x_1, x_2, x_3]	[x_2, x_3, x_4]

⸻

4.3 Teacher Forcing

Ground truth tokens are used as inputs during training, ensuring stable gradients.

⸻

5. Inference (Autoregressive Generation)

5.1 Process

1. Initialize with input sequence
2. Predict next token:
    x_t = \arg\max P(x_t \mid x_{<t})
3. Append prediction to sequence
4. Repeat until <EOS> or max length

⸻

5.2 Greedy Decoding

Uses:
\arg\max \text{logits}

Alternative strategies:

* beam search
* sampling

⸻

6. Computational Characteristics

6.1 Time Complexity

Self-attention:
O(n^2 \cdot d_{model})

Due to pairwise token interactions.

⸻

6.2 Memory Complexity

O(n^2)

For attention matrix storage.

⸻

7. Limitations of Minimal Implementation

Your implementation omits:

* Layer Normalization
* Feed Forward Network (FFN)
* Multi-Head Attention
* Dropout

These are essential in production-scale models.

⸻

8. Advantages

* Parallel computation (during training)
* Global context modeling
* Simpler architecture than encoder–decoder

⸻

9. Use Cases

* Language modeling
* Text generation
* Code generation
* Conversational AI

⸻

10. Summary

A decoder-only transformer:

* models sequences autoregressively
* uses masked self-attention to enforce causality
* predicts tokens step-by-step
* scales efficiently for large language models

It forms the foundation of modern generative systems by learning:

P(x_1, x_2, ..., x_n) = \prod_{t=1}^{n} P(x_t \mid x_{<t})

⸻

If you want a deeper layer, I can extend this into:

* full GPT block (with LayerNorm + FFN)
* KV caching optimization
* or mathematical derivation of multi-head attention.